In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModel,
    pipeline as hf_pipeline
)
from sentence_transformers import SentenceTransformer, util as st_util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load train.csv using Hugging Face datasets
dataset = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')['train']

Generating train split: 0 examples [00:00, ? examples/s]

Q1: combined_text for row index 51

In [3]:
dataset = dataset.map(lambda x: {'combined_text': x['prompt'] + ' ' + x['A']})
print("Q1: Character length of combined_text at index 51")
print(len(dataset[51]['combined_text']), "\n")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Q1: Character length of combined_text at index 51
614 



Q2: BERT vocab size

In [4]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
print("Q2: BERT vocab size")
print(tokenizer.vocab_size, "\n")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q2: BERT vocab size
30522 



Q3: [SEP] token ID

In [5]:
print("Q3: [SEP] token ID")
print(tokenizer.sep_token_id, "\n")

Q3: [SEP] token ID
102 



Q4: Shape of input_ids tensor (all prompts, max_length=128)

In [6]:
prompts = list(dataset['prompt'])
encoded = tokenizer(
    prompts,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)
print("Q4: Shape of input_ids tensor")
print(encoded['input_ids'].shape, "\n")

Q4: Shape of input_ids tensor
torch.Size([2000, 128]) 



Q5: Dimensionality of each attention head

In [7]:
print("Q5: Attention head dimensionality")
print(768 // 12, "\n")

Q5: Attention head dimensionality
64 



Q6: Shape of last_hidden_state for row ID 0

In [8]:
model = AutoModel.from_pretrained('bert-base-uncased')
model.eval()

row0_prompt = dataset[0]['prompt']
inputs = tokenizer(row0_prompt, return_tensors='pt')
with torch.no_grad():
    outputs = model(**inputs)

print("Q6: Shape of last_hidden_state for row ID 0")
print(outputs.last_hidden_state.shape, "\n")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q6: Shape of last_hidden_state for row ID 0
torch.Size([1, 31, 768]) 



Q7: Sum of first 5 floats in [CLS] vector

In [9]:
cls_vector = outputs.last_hidden_state[0, 0, :]
print("Q7: Sum of first 5 values in [CLS] vector")
print(round(cls_vector[:5].sum().item(), 4), "\n")

Q7: Sum of first 5 values in [CLS] vector
-1.2001 



Q8: Attention weight [CLS] → "fusion" in last layer, head 0

In [10]:
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
model_attn.eval()

text = "Light-ion fusion is a technique."
inputs2 = tokenizer(text, return_tensors='pt')
with torch.no_grad():
    outputs2 = model_attn(**inputs2)

tokens = tokenizer.convert_ids_to_tokens(inputs2['input_ids'][0])
print("Tokens:", tokens)
fusion_idx = tokens.index('fusion')
attn_weight = outputs2.attentions[-1][0, 0, 0, fusion_idx].item()
print("Q8: Attention weight [CLS] → 'fusion' (last layer, head 0)")
print(round(attn_weight, 4), "\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Q8: Attention weight [CLS] → 'fusion' (last layer, head 0)
0.1025 



Q9: Cosine similarity prompt vs Option B for row ID 0 (MiniLM)

In [11]:
mini_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
row0 = dataset[0]
emb_prompt = mini_model.encode(row0['prompt'], convert_to_tensor=True)
emb_b      = mini_model.encode(row0['B'],      convert_to_tensor=True)
sim = st_util.cos_sim(emb_prompt, emb_b).item()
print("Q9: Cosine similarity (prompt vs Option B, row 0)")
print(round(sim, 4), "\n")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Q9: Cosine similarity (prompt vs Option B, row 0)
0.7658 



Q10: MAP@3 — TF-IDF vs MiniLM pipelines

In [12]:
OPTIONS = ['A', 'B', 'C', 'D', 'E']

def map_at_3(truth, preds):
    for i, p in enumerate(preds[:3]):
        if p == truth:
            return 1.0 / (i + 1)
    return 0.0

# TF-IDF pipeline (from Milestone 1)
combined_texts = [
    row['prompt'] + ' ' + ' '.join([str(row[o]) for o in OPTIONS])
    for row in dataset
]
tfidf_vec = TfidfVectorizer(stop_words='english')
tfidf_vec.fit(combined_texts)

tfidf_preds = []
for row in dataset:
    pv = tfidf_vec.transform([row['prompt']])
    sims = {o: cosine_similarity(pv, tfidf_vec.transform([str(row[o])]))[0][0] for o in OPTIONS}
    tfidf_preds.append(sorted(OPTIONS, key=lambda o: sims[o], reverse=True)[:3])

# MiniLM pipeline
print("Running MiniLM pipeline (this may take a while)...")
all_prompts = dataset['prompt']
all_option_texts = {o: dataset[o] for o in OPTIONS}

prompt_embs = mini_model.encode(all_prompts, batch_size=64, show_progress_bar=True, convert_to_tensor=True)
option_embs = {o: mini_model.encode(all_option_texts[o], batch_size=64, show_progress_bar=False, convert_to_tensor=True) for o in OPTIONS}

mini_preds = []
for i in range(len(dataset)):
    sims = {o: st_util.cos_sim(prompt_embs[i], option_embs[o][i]).item() for o in OPTIONS}
    mini_preds.append(sorted(OPTIONS, key=lambda o: sims[o], reverse=True)[:3])

answers = dataset['answer']
tfidf_scores = [map_at_3(answers[i], tfidf_preds[i]) for i in range(len(dataset))]
mini_scores  = [map_at_3(answers[i], mini_preds[i])  for i in range(len(dataset))]

print("Q10a: MiniLM MAP@3")
print(round(np.mean(mini_scores), 4), "\n")

# Count: correct NOT in TF-IDF top-3 but IS in MiniLM top-3
count = sum(
    1 for i in range(len(dataset))
    if answers[i] not in tfidf_preds[i] and answers[i] in mini_preds[i]
)
print("Q10b: Questions where MiniLM finds answer but TF-IDF doesn't")
print(count, "\n")

Running MiniLM pipeline (this may take a while)...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Q10a: MiniLM MAP@3
0.4231 

Q10b: Questions where MiniLM finds answer but TF-IDF doesn't
502 



Q11: Zero-shot classification (index 1)

In [13]:
row1 = dataset[1]
zsc = hf_pipeline('zero-shot-classification')  # defaults to facebook/bart-large-mnli
result = zsc(row1['prompt'], candidate_labels=[row1['A'], row1['B'], row1['C']])
print("Q11: Zero-shot top-ranked probability")
print(round(result['scores'][0], 4))
print("All scores:", [round(s, 4) for s in result['scores']], "\n")

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q11: Zero-shot top-ranked probability
0.4575
All scores: [0.4575, 0.2751, 0.2675] 



Q12: Zero-shot with multi_label=True 

In [14]:
result_multi = zsc(row1['prompt'], candidate_labels=[row1['A'], row1['B'], row1['C']], multi_label=True)
sum_softmax  = sum(result['scores'])
sum_sigmoid  = sum(result_multi['scores'])
diff = abs(sum_softmax - sum_sigmoid)
print("Q12: Absolute difference between softmax sum and sigmoid sum")
print(f"Softmax sum: {round(sum_softmax, 4)}")
print(f"Sigmoid sum: {round(sum_sigmoid, 4)}")
print(f"Absolute difference: {round(diff, 4)}\n")

Q12: Absolute difference between softmax sum and sigmoid sum
Softmax sum: 1.0
Sigmoid sum: 0.0005
Absolute difference: 0.9995



Q13: Flan-T5-small generation for row index 0 

In [15]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

row0 = dataset[0]

flan_tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-small')
flan_model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-small')

flan_input = (
    f"Question: {row0['prompt']}. "
    f"Is the correct answer A: {row0['A']} or B: {row0['B']}? "
    f"Answer with just the letter A or B."
)

input_ids = flan_tokenizer(flan_input, return_tensors='pt').input_ids
outputs = flan_model.generate(input_ids, max_new_tokens=5)
result = flan_tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Q13: Flan-T5-small output")
print(repr(result))

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Q13: Flan-T5-small output
'B'
